# Pipeline treningowy — predykcja czasu półmaratonu

Wejście dostępne dla użytkownika aplikacji: **płeć, wiek, czas na 5 km**.
Model uczymy więc tylko na tych trzech cechach → target `Czas` (cały półmaraton),
żeby uniknąć data leakage (cechy typu 10/15/20 km, tempo, miejsce nie są znane
przed startem).

Dane wczytujemy z Digital Ocean Spaces (wgrane w poprzednim etapie).

In [ ]:
import pandas as pd
from spaces_client import read_csv_from_spaces, upload_file

## 1. Wczytanie danych z Digital Ocean Spaces

In [ ]:
RACE_FILES = {
    2023: "data/halfmarathon_wroclaw_2023__final.csv",
    2024: "data/halfmarathon_wroclaw_2024__final.csv",
}

frames = []
for race_year, spaces_key in RACE_FILES.items():
    df_year = read_csv_from_spaces(spaces_key, sep=";")
    df_year["race_year"] = race_year
    frames.append(df_year)

df = pd.concat(frames, ignore_index=True)
df.shape

## 2. Czyszczenie danych

In [ ]:
def time_to_seconds(time_str):
    if pd.isnull(time_str) or time_str in ("DNS", "DNF"):
        return None
    h, m, s = time_str.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


df["time_5k_s"] = df["5 km Czas"].apply(time_to_seconds)
df["time_total_s"] = df["Czas"].apply(time_to_seconds)
df["age"] = df["race_year"] - df["Rocznik"]
df["gender"] = df["Płeć"]

In [ ]:
model_cols = ["gender", "age", "time_5k_s", "time_total_s"]
df_model = df[model_cols].copy()

print("Braki danych (%):")
print((df_model.isnull().sum() / len(df_model) * 100).round(2))

df_model = df_model.dropna()
df_model = df_model[df_model["gender"].isin(["M", "K"])]
df_model = df_model[(df_model["age"] >= 15) & (df_model["age"] <= 90)]
df_model.shape

In [ ]:
# Usunięcie odstających wartości czasu całkowitego (IQR)
Q1 = df_model["time_total_s"].quantile(0.25)
Q3 = df_model["time_total_s"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

df_model = df_model[
    (df_model["time_total_s"] >= lower) & (df_model["time_total_s"] <= upper)
]
df_model.shape

## 3. Feature selection

Cechy ograniczone do tego, co użytkownik może podać w aplikacji: `gender`, `age`,
`time_5k_s`. Sprawdzamy korelację `time_5k_s` z targetem jako uzasadnienie —
oczekujemy silnej korelacji (czas na 5 km jest dobrym predyktorem całości).

In [ ]:
df_model[["age", "time_5k_s", "time_total_s"]].corr()["time_total_s"]

## 4. Trening modelu (PyCaret — regresja)

In [ ]:
from pycaret.regression import (
    setup, compare_models, tune_model, finalize_model,
    save_model, predict_model, pull,
)

exp_setup = setup(
    data=df_model,
    target="time_total_s",
    session_id=123,
    train_size=0.8,
    verbose=False,
)

In [ ]:
best_models = compare_models(sort="RMSE", n_select=3)
compare_metrics = pull()
compare_metrics

In [ ]:
tuned_best = tune_model(best_models[0], optimize="RMSE", choose_better=True)

In [ ]:
final_model = finalize_model(tuned_best)
predict_model(final_model)  # wynik na holdout
holdout_metrics = pull()
holdout_metrics

## 5. Zapis modelu — lokalnie i do Digital Ocean Spaces

In [ ]:
MODEL_NAME = "polmaraton_model"

save_model(final_model, MODEL_NAME)  # zapisuje lokalnie jako polmaraton_model.pkl

model_url = upload_file(f"{MODEL_NAME}.pkl", spaces_key=f"models/{MODEL_NAME}.pkl")
model_url